# Finanças pessoais com redes neurais — Google Colab

Este notebook recebe o `projeto.zip` criado por `python tools/build_colab_zip.py`. Não precisa montar o Drive, clonar o GitHub ou ajustar caminhos: execute as células em ordem e selecione o ZIP quando solicitado.

O fluxo padrão roda um teste rápido com 200 usuários. A execução completa (1.000 usuários e três seeds) é opcional e pode levar mais tempo. Baixe o pacote de resultados antes de fechar a sessão, pois o disco do Colab é temporário.

In [ ]:
# 1. Envie o projeto.zip gerado localmente com tools/build_colab_zip.py.
from google.colab import files
from pathlib import Path, PurePosixPath
import io
import shutil
import zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f'Envie exatamente um arquivo .zip; recebidos: {zip_names}')

PROJECT = Path('/content/projeto')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
    for item in archive.infolist():
        member = PurePosixPath(item.filename)
        destination = (PROJECT / Path(*member.parts)).resolve()
        if member.is_absolute() or '..' in member.parts or PROJECT.resolve() not in destination.parents:
            raise ValueError(f'Caminho inválido no ZIP: {item.filename}')
    archive.extractall(PROJECT)

if not (PROJECT / 'src' / 'rodar_tudo.py').is_file():
    raise FileNotFoundError('ZIP inválido: src/rodar_tudo.py não foi encontrado')
%cd /content/projeto

In [ ]:
# 2. Instale as dependências do projeto. O runtime Colab normalmente já inclui PyTorch.
%pip -q install -r requirements.txt
import sys, torch, pandas, sklearn, mlflow
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| pandas', pandas.__version__)
print('scikit-learn', sklearn.__version__, '| MLflow', mlflow.__version__, '| GPU:', torch.cuda.is_available())

In [ ]:
# 3. Mantenha os resultados no disco local e temporário da sessão do Colab.
import os
PROJECT = Path('/content/projeto')
os.environ['MLFLOW_TRACKING_URI'] = 'sqlite:////content/projeto/results/mlflow.db'
os.environ['MLFLOW_ARTIFACT_ROOT'] = '/content/projeto/results/mlartifacts'
(PROJECT / 'results').mkdir(parents=True, exist_ok=True)

In [ ]:
# 4. Teste rápido: gera 200 usuários, roda os baselines e treina as redes por poucas épocas.
!python src/rodar_tudo.py --rapido --mlflow --seeds 0

## Execução completa (opcional)

Depois do teste rápido, execute a célula abaixo para gerar novamente os dados com 1.000 usuários e repetir os treinos com três seeds. `--regerar` garante que o conjunto pequeno do teste rápido não seja reutilizado.

In [ ]:
# Remova o comentário e execute para rodar o experimento completo:
# !python src/rodar_tudo.py --mlflow --usuarios 1000 --seeds 0 1 2 --regerar

In [ ]:
# 5. Consulte os runs e as métricas registrados no MLflow.
import mlflow
import pandas as pd

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
runs = mlflow.search_runs(
    experiment_names=['anomalias', 'previsao-gastos'],
    order_by=['start_time DESC'],
)
metric_cols = [c for c in runs.columns if c.startswith('metrics.teste/') and
               ('PR-AUC' in c or 'wape_geral' in c or 'Recall_1' in c)]
display(runs[['tags.mlflow.runName', 'tags.modelo', 'tags.data.md5'] + metric_cols].head(30))

In [ ]:
# 6. Baixe os resultados antes de encerrar o runtime.
from google.colab import files
import shutil

runs.to_csv(PROJECT / 'results' / 'runs.csv', index=False)
result_zip = shutil.make_archive('/content/financas_nn_resultados', 'zip', root_dir=PROJECT,
                                 base_dir='results')
files.download(result_zip)

O pacote baixado contém o banco MLflow e os artefatos produzidos durante a sessão. O CSV `results/runs.csv` facilita a leitura das métricas; para abrir o banco MLflow com os artefatos, extraia o pacote de volta em `/content/projeto`, pois os caminhos registrados no banco apontam para essa pasta do Colab.